# LSTM model

## Install libraries

In [1]:
# Standard libraries
import os
import random
import sys

# Third-party libraries
import ipynbname
import lightning as L
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dotenv import load_dotenv
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset

# Project path setup
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

# Custom libraries
from dtos.config_dtos.config_dto import ConfigDto
from logger.logger import Logger
from utils.constants import *
from utils.utils import *
from result_evaluator.result_evaluator import ResultEvaluator

# Torch serialization safe globals
torch.serialization.add_safe_globals(
    [ScalerType, ModelAchitectureType, OptimizerType, LossFunctionType]
)

In [2]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Parameters

In [3]:
NOTEBOOK_NAME = ipynbname.name()
RANDOM_SEED = 18

STOCK_CODE = "VIC"
FEATURE_GROUPS = ["basic", "add_ad"]
LOOKBACK_WINDOW = 50
FORECAST_HORIZON = 5
ID_COLUMN = ["date", "code"]
TARGET_COLUMN = f"adjust"

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VAL_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-04-30")



## Load data

In [4]:
TRAIN_TEST_SET_STOCK_CODE = str.lower(
    f"../{TRAIN_TEST_SET_DIR}/{STOCK_CODE}_{LOOKBACK_WINDOW}_{FORECAST_HORIZON}"
)
TRAIN_TEST_SET_STOCK_CODE

'../../../train_test_set/vic_50_5'

In [5]:
# Load feature datasets
X_train = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/X_train.npy")
X_val = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/X_val.npy")
X_test = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/X_test.npy")

# Load label datasets
y_train = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/y_train.npy")
y_val = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/y_val.npy")
y_test = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/y_test.npy")

# Load dates
dates_train = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_train.npy")
dates_val = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_val.npy")
dates_test = np.load(f"{TRAIN_TEST_SET_STOCK_CODE}/dates_test.npy")

# Optional: check shapes
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

print("dates_train:", dates_train.shape)
print("dates_val:", dates_val.shape)
print("dates_test:", dates_test.shape)

X_train: (3459, 50, 70)
X_val: (439, 50, 70)
X_test: (510, 50, 70)
y_train: (3459,)
y_val: (439,)
y_test: (510,)
dates_train: (3459,)
dates_val: (439,)
dates_test: (510,)


## Create model

### Hyperparameters

In [6]:
MAX_EPOCHS = 15
LEARNING_RATE = 1.0e-03
PATIENCE = 5
BATCH_SIZE = 64

In [7]:
MODEL_ADDITIONAL_PARAMS = {}

### Data Loader

In [8]:
class StockDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [9]:
train_ds = StockDataset(X_train, y_train)
val_ds = StockDataset(X_val, y_val)
test_ds = StockDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

In [10]:
INPUT_SIZE = X_train.shape[2]
print(f"Input size: {INPUT_SIZE}")

Input size: 70


### Config Model

In [11]:
config_dto = ConfigDto(
    # Data
    notebook_name=NOTEBOOK_NAME,
    feature_groups=FEATURE_GROUPS,
    stock_code=STOCK_CODE,
    train_start_date=TRAIN_RANGE[0],
    train_end_date=TRAIN_RANGE[1],
    validation_start_date=VAL_RANGE[0],
    validation_end_date=VAL_RANGE[1],
    test_start_date=TEST_RANGE[0],
    test_end_date=TEST_RANGE[1],
    # Data Hyperparameters
    random_seed=RANDOM_SEED,
    lookback_window_size=LOOKBACK_WINDOW,
    forecast_window_size=FORECAST_HORIZON,
    scaler_type=ScalerType.STANDARD,
    # Model
    model_architecture=ModelAchitectureType.LSTM,
    model_additional_params=MODEL_ADDITIONAL_PARAMS,
    # Training Hyperparameters
    epochs=MAX_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    optimizer=OptimizerType.ADAM,
    loss_fn=LossFunctionType.MSE,
    patience=PATIENCE,
)

### Model

In [12]:
class LightningLSTM(L.LightningModule):
    def __init__(self, random_seed, input_size, config_dto: ConfigDto):
        super().__init__()

        L.seed_everything(seed=random_seed)

        self.input_size = int(input_size)
        if self.input_size <= 0:
            raise ValueError("input_size must be greater than 0")

        self.hidden_size = 128
        self.num_layers = 2

        # input_size must match the last dimension of X: 70
        self.lstm = nn.LSTM(
            input_size=self.input_size,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True,
            dropout=0.2 if self.num_layers > 1 else 0.0,
        )

        self.fc = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

        self.save_hyperparameters(asdict(config_dto))

    def forward(self, x):
        # x shape: [B, 50, 70]
        lstm_out, _ = self.lstm(x)  # [B, 50, hidden_size]
        x = lstm_out[:, -1, :]  # last timestep -> [B, hidden_size]
        x = self.fc(x)  # [B, 1]
        return x.squeeze(-1)  # [B]

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=LEARNING_RATE)

    def training_step(self, batch, batch_idx):
        input_i, label_i = batch
        label_i = label_i.float()

        output_i = self(input_i)
        loss = F.mse_loss(output_i, label_i)

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_i, label_i = batch
        label_i = label_i.float()

        output_i = self(input_i)
        loss = F.mse_loss(output_i, label_i)

        self.log("val_loss", loss, prog_bar=True)
        return loss

In [13]:
model = LightningLSTM(RANDOM_SEED, INPUT_SIZE, config_dto)

Seed set to 18


### Train model

In [14]:
early_stop_callback = EarlyStopping(
    monitor="val_loss",  # metric to watch
    patience=PATIENCE,  # epochs to wait before stopping
    mode="min",  # because lower loss is better
)

In [15]:
# logger = TensorBoardLogger(
#     "D:/GIT/master-thesis/src/model/lightning_logs", name="", version=3
# )
# path_to_checkpoint = "D:/GIT/master-thesis/src/model/lightning_logs/version_3/checkpoints/epoch=4960-step=783838.ckpt"

logger = None
path_to_checkpoint = None

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    log_every_n_steps=1,
    callbacks=[early_stop_callback],
    logger=logger,
)

trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path=path_to_checkpoint,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name | Type       | Params | Mode  | FLOPs
----------------------------------------------------
0 | lstm | LSTM       | 234 K  | train | 0    
1 | fc   | Sequential | 74.2 K | train | 0    
----------------------------------------------------
308 K     Trainable params
0         Non-trainable params
308 K     Total params
1.235     Total estimated model params size (MB)
9         Modules in train mode
0         Modules in eval mode
0         Total Flops


d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
d:\GIT\master-thesis\mt_env\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Epoch 6: 100%|██████████| 55/55 [00:02<00:00, 19.46it/s, v_num=1, train_loss=0.000179, val_loss=0.00316]


### Evaluate model

In [16]:
y_test[:10], y_test.shape

(array([1.01605206, 1.00085106, 1.0116129 , 1.02365591, 1.0223176 ,
        1.02134927, 1.01615646, 1.01530612, 1.00504202, 1.00335852]),
 (510,))

In [17]:
y_pred = model(torch.tensor(X_test, dtype=torch.float32))
y_pred = y_pred.detach().numpy()

y_pred[:10], y_pred.shape

(array([0.98697585, 1.0079901 , 1.0361466 , 1.0368958 , 1.0194902 ,
        1.0327075 , 1.0105581 , 1.004699  , 1.0280931 , 1.0195503 ],
       dtype=float32),
 (510,))

In [18]:
dates_test[:10], dates_test.shape

(array(['2024-03-18T00:00:00.000000000', '2024-03-19T00:00:00.000000000',
        '2024-03-20T00:00:00.000000000', '2024-03-21T00:00:00.000000000',
        '2024-03-22T00:00:00.000000000', '2024-03-25T00:00:00.000000000',
        '2024-03-26T00:00:00.000000000', '2024-03-27T00:00:00.000000000',
        '2024-03-28T00:00:00.000000000', '2024-03-29T00:00:00.000000000'],
       dtype='datetime64[ns]'),
 (510,))

In [19]:
result_evaluator = ResultEvaluator(y_pred, y_test, dates_test)

In [20]:
valuator = ResultEvaluator(y_pred, y_test, dates_test)

In [21]:
result_evaluator.evaluate()

,metric,value
0,MAE,0.074276
1,MSE,0.011167
2,RMSE,0.105674
3,MAPE,0.069308
4,R2,-0.935440


In [22]:
result_folder = get_current_run_path()
result_folder

WindowsPath('lightning_logs/version_1')

In [23]:
result_evaluator.save_result(result_folder)

In [1]:
import torch
print(torch.cuda.is_available())      # must be True
print(torch.cuda.get_device_name(0))  # NVIDIA GeForce RTX 3050

True
NVIDIA GeForce RTX 3050 Laptop GPU
